# Step 7: Better Forecasts — Bias, Recalibration, Forecast Head

## Why your skill scores looked terrible

1. Eval scored **raw PINN** without the **bias correction** `forecaster.py` already uses
2. The model was trained as an **interpolator** `(lat, lon, time) → T`, not a forecaster
3. Newest years ran **~0.4°C warm** (drift) — needs recalibration

This notebook fixes those three issues step by step.

## Part A — Bias-aware evaluation (quick win)

At issue time `t−h` we compute:

`bias = mean(actual − PINN)` over the last 7 days

then predict `PINN(t) + bias`.

This matches production. Re-score vs persistence / climatology.

In [ ]:
import importlib
import validate_forecast
importlib.reload(validate_forecast)
from validate_forecast import run as run_forecast_validation

print("Running bias-aware forecast validation...")
import os
forecast_exists = os.path.exists("pinn_forecast_best.h5")
print("Forecast head present:", forecast_exists)
print("(If forecast head exists it will be used; else interpolating PINN + bias)")

out = run_forecast_validation()
out["summary"]


## Part B — Rolling recalibration (fix warm test bias)

Fit additive offsets on the **validation** split only (no test leakage):

`offset[location] = mean(actual − PINN)` on val

Saved to `recalibration.pkl` and applied by the forecaster + eval.

In [2]:
from recalibrate import recalibrate_and_save

recal = recalibrate_and_save(source="val")
recal["offsets"]

RECALIBRATION OFFSETS (°C)  [actual - pinn]
source : val  (n=2010)
model  : d:\corals\model\pinn_model_best.h5
  global      : +0.0709 °C
  hikkaduwa   : +0.0668 °C
  kalpitiya   : +0.1912 °C
  passikudha  : +0.0146 °C
  south_east  : +0.1093 °C
  trinco      : -0.0276 °C
Saved -> d:\corals\model\recalibration.pkl


{'global': 0.07085284074622014,
 'hikkaduwa': 0.06678610749505644,
 'kalpitiya': 0.19116322607543337,
 'passikudha': 0.014598430045208466,
 'south_east': 0.10929112809214435,
 'trinco': -0.027574687976742226}

In [3]:
# Re-run validation with recalibration loaded
out2 = run_forecast_validation()
out2["summary"]

FORECAST VALIDATION (MAE °C, lower is better)
Model: d:\corals\model\pinn_model_best.h5  (forecast_head=False)
Recalibration offsets: {'global': 0.07085284074622014, 'hikkaduwa': 0.06678610749505644, 'kalpitiya': 0.19116322607543337, 'passikudha': 0.014598430045208466, 'south_east': 0.10929112809214435, 'trinco': -0.027574687976742226}
 horizon_days  pinn_raw_mae  pinn_bias_mae  pinn_recal_mae  persistence_mae  climatology_mae  skill_bias_vs_persistence  skill_bias_vs_climatology
            1      0.872495       0.215793        0.231262         0.154264         0.413547                  -0.398855                   0.478190
            3      0.872495       0.276405        0.289333         0.243756         0.413547                  -0.133941                   0.331622
            7      0.872495       0.368569        0.378083         0.342522         0.413547                  -0.076044                   0.108760

SUCCESS CRITERIA (honest bars)
  1-day : do NOT need to beat persistence 

,horizon_days,pinn_raw_mae,pinn_bias_mae,pinn_recal_mae,persistence_mae,climatology_mae,pinn_raw_rmse,pinn_bias_rmse,pinn_recal_rmse,persistence_rmse,climatology_rmse,n,skill_bias_vs_persistence,skill_bias_vs_climatology,skill_raw_vs_persistence,skill_recal_vs_persistence,pass_beat_persistence,pass_beat_climatology
0,1,0.872495,0.215793,0.231262,0.154264,0.413547,1.093308,0.282761,0.299520,0.211664,0.513474,2010,-0.398855,0.478190,-4.655865,-0.499137,None,True
1,3,0.872495,0.276405,0.289333,0.243756,0.413547,1.093308,0.365501,0.379935,0.319820,0.513474,2010,-0.133941,0.331622,-2.579374,-0.186979,False,True
2,7,0.872495,0.368569,0.378083,0.342522,0.413547,1.093308,0.481602,0.494861,0.436397,0.513474,2010,-0.076044,0.108760,-1.547263,-0.103819,False,True


## Part C — Train a real forecast head (+1 / +3 / +7 days)

Build supervised samples:

```
X = [lat, lon, time_target, horizon, temp_at_issue, dhw_at_issue]
y = [temp_at_target, dhw_at_target]
```

This is the loss that matches how we evaluate forecasts.

In [4]:
from prepare_forecast_data import prepare_forecast_dataset

meta = prepare_forecast_dataset()
meta

Forecast dataset prepared:
  horizons: [1, 3, 7]
  input_features: ['lat_norm', 'lon_norm', 'time_norm_target', 'horizon_norm', 'temp_norm_issue', 'dhw_norm_issue']
  output_features: ['temp_norm_target', 'dhw_norm_target']
  n_train: 48170
  n_val: 5975
  n_test: 5975
  X_train_shape: [48170, 6]
  y_train_shape: [48170, 2]


{'horizons': [1, 3, 7],
 'input_features': ['lat_norm',
  'lon_norm',
  'time_norm_target',
  'horizon_norm',
  'temp_norm_issue',
  'dhw_norm_issue'],
 'output_features': ['temp_norm_target', 'dhw_norm_target'],
 'n_train': 48170,
 'n_val': 5975,
 'n_test': 5975,
 'X_train_shape': [48170, 6],
 'y_train_shape': [48170, 2]}

In [5]:
from train_forecast import train

# Shorter run for a first pass; increase epochs later if needed
history = train(epochs=80, batch_size=128)
print("Best val loss:", min(history.history["val_loss"]))

Forecast model uses MSE on SST+DHW (no PDE on 6-D inputs)
X_train (48170, 6)  y_train (48170, 2)
X_val   (5975, 6)  y_val   (5975, 2)
Epoch 1/80
365/377 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0561 - mae: 0.1688


Epoch 1: val_loss=0.006228 -> pinn_forecast_best.h5
377/377 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.0277 - mae: 0.1122 - val_loss: 0.0062 - val_mae: 0.0537 - learning_rate: 0.0010
Epoch 2/80
370/377 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0033 - mae: 0.0476


Epoch 2: val_loss=0.005579 -> pinn_forecast_best.h5
377/377 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0027 - mae: 0.0424 - val_loss: 0.0056 - val_mae: 0.0496 - learning_rate: 0.0010
Epoch 3/80
375/377 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0019 - mae: 0.0343


Epoch 3: val_loss=0.004127 -> pinn_forecast_best.h5
377/377 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0018 - mae: 0.0324 - val_loss: 0.0041 - val_mae: 0.0394 - learning_rate: 0.0010
Epoch 4/80
371/377 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0015 - mae: 0.0288


Epoch 4: val_loss=0.003311 -> pinn_forecast_best.h5
377/377 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0015 - mae: 0.0282 - val_loss: 0.0033 - val_mae: 0.0358 - learning_rate: 0.0010
Epoch 5/80
353/377 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0014 - mae: 0.0268

...


Epoch 6: val_loss=0.002200 -> pinn_forecast_best.h5
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0013 - mae: 0.0250 - val_loss: 0.0022 - val_mae: 0.0297 - learning_rate: 0.0010
Epoch 7/80
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0013 - mae: 0.0240 - val_loss: 0.0022 - val_mae: 0.0295 - learning_rate: 0.0010
Epoch 8/80
373/377 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0013 - mae: 0.0240


Epoch 8: val_loss=0.002050 -> pinn_forecast_best.h5
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0013 - mae: 0.0235 - val_loss: 0.0021 - val_mae: 0.0286 - learning_rate: 0.0010
Epoch 9/80
368/377 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0012 - mae: 0.0228


Epoch 9: val_loss=0.001989 -> pinn_forecast_best.h5
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0012 - mae: 0.0228 - val_loss: 0.0020 - val_mae: 0.0283 - learning_rate: 0.0010
Epoch 10/80
366/377 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0012 - mae: 0.0225


Epoch 10: val_loss=0.001801 -> pinn_forecast_best.h5
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0012 - mae: 0.0224 - val_loss: 0.0018 - val_mae: 0.0273 - learning_rate: 0.0010
Epoch 11/80
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0012 - mae: 0.0223 - val_loss: 0.0019 - val_mae: 0.0274 - learning_rate: 0.0010
Epoch 12/80
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0012 - mae: 0.0220 - val_loss: 0.0020 - val_mae: 0.0293 - learning_rate: 0.0010
Epoch 13/80
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0012 - mae: 0.0218 - val_loss: 0.0028 - val_mae: 0.0298 - learning_rate: 0.0010
Epoch 14/80
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0012 - mae: 0.0218 - val_loss: 0.0030 - val_mae: 0.0308 - learning_rate: 0.0010
Epoch 15/80
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0012 - mae: 0.0215 - val_loss: 0.0022 - val_mae: 0.0286 - learning_rate: 0.0010
Epoch 16/80
377/377 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0012 - mae: 0.0212 - val_loss: 0.0025

Saved pinn_forecast_best.h5 / pinn_forecast_final.h5
History -> d:\corals\model\forecast_training_history.csv
Best val loss: 0.0018011156935244799


## Part D — Re-evaluate with the forecast head

`validate_forecast.py` prefers `pinn_forecast_best.h5` when it exists.

In [9]:
import importlib
import validate_forecast
importlib.reload(validate_forecast)
from validate_forecast import run as run_forecast_validation

out3 = run_forecast_validation()
summary = out3["summary"]
summary


FORECAST VALIDATION (MAE °C, lower is better)
Model: d:\corals\model\pinn_forecast_best.h5  (forecast_head=True)
Recalibration offsets: {'global': 0.07085284074622014, 'hikkaduwa': 0.06678610749505644, 'kalpitiya': 0.19116322607543337, 'passikudha': 0.014598430045208466, 'south_east': 0.10929112809214435, 'trinco': -0.027574687976742226}
 horizon_days  pinn_raw_mae  pinn_bias_mae  pinn_recal_mae  persistence_mae  climatology_mae  skill_bias_vs_persistence  skill_bias_vs_climatology
            1      0.201677       0.201677        0.227388         0.154264         0.413547                  -0.307353                   0.512323
            3      0.286295       0.286295        0.309061         0.243756         0.413547                  -0.174512                   0.307709
            7      0.371101       0.371101        0.406565         0.342522         0.413547                  -0.083434                   0.102639

SUCCESS CRITERIA (honest bars)
  1-day : do NOT need to beat persistenc

,horizon_days,pinn_raw_mae,pinn_bias_mae,pinn_recal_mae,persistence_mae,climatology_mae,pinn_raw_rmse,pinn_bias_rmse,pinn_recal_rmse,persistence_rmse,climatology_rmse,n,skill_bias_vs_persistence,skill_bias_vs_climatology,skill_raw_vs_persistence,skill_recal_vs_persistence,pass_beat_persistence,pass_beat_climatology
0,1,0.201677,0.201677,0.227388,0.154264,0.413547,0.256931,0.256931,0.282124,0.211664,0.513474,2010,-0.307353,0.512323,-0.307353,-0.474022,None,True
1,3,0.286295,0.286295,0.309061,0.243756,0.413547,0.360457,0.360457,0.385931,0.319820,0.513474,2010,-0.174512,0.307709,-0.174512,-0.267910,False,True
2,7,0.371101,0.371101,0.406565,0.342522,0.413547,0.469971,0.469971,0.510456,0.436397,0.513474,2010,-0.083434,0.102639,-0.083434,-0.186973,False,True


### How to read the new bars

| Horizon | Healthy goal |
|---------|--------------|
| 1-day | Persistence often wins — OK |
| 3–7 day | `skill_bias_vs_persistence > 0` |
| All | Beat climatology |

**Next:** `08_tune_physics.ipynb` then re-train interpolating PINN in `02`.